# Radiance to reflectance — SWIR coffee/chicory plates

Reads each `.hdr` / `.img` pair, converts radiance to reflectance against the **white** Spectralon strip
only, and writes out per-plate reflectance with rounded wavelength labels.

**Scans**

| Day | Plates | Order from the reference strip |
|---|---|---|
| 1 | 6 | AB100, CH100, RB100, A, B, C |
| 2 | 5 | RB[50:50], RB[60:40], RB[70:30], RB[80:20], RB[90:10] |

RB = robusta, CH = chicory, `100` = pure, `RB[70:30]` = 70 % robusta / 30 % chicory. A, B and C are the
unknowns.

**The conversion**

    R(row, col, band) = L(row, col, band) / W(col, band) × R_certified_white(band)

Two details that matter:

*The white reference is built **per across-track column**.* Your strip spans the full 384-pixel swath, so
the lamp and lens profile can be measured at every column instead of assumed. That matters here because
the plates sit at different across-track positions from scan to scan — dividing by one averaged white
spectrum would leave that profile baked into the plates as a spectral error.

*The certificate is not flat 0.99.* The white Spectralon falls from 0.992 at 1000 nm to 0.948 by 2450 nm.
Treating it as constant would put a systematic ~4 % error at the long-wavelength end, which is exactly
where the C-H bands that distinguish coffee from chicory sit. The certificate is interpolated onto the
camera's band centres.

**One thing you are giving up by using only the white strip.** With a second strip you could solve for an
additive offset (residual dark current and stray light) and verify the reflectance scale independently.
With one strip the offset is assumed zero and nothing checks the result. If the output looks off later,
that discarded strip is the first place to look.

**Header facts used here:** 384 samples × 2000 lines × 288 bands, BIL, float32, little-endian,
`header offset = 2666496`, `data ignore value = 2`, 952.16–2516.63 nm at ~5.45 nm spacing.

## 1. Configuration

In [ ]:
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "figure.facecolor": "white"})

# ------------------------------------ EDIT ------------------------------------
DATA_DIR  = Path(os.environ.get("COFFEE_RAW", "/run/media/riyanvarghese/RIYAN HDD"))   # folder with the .hdr / .img pairs
PANEL_TXT = Path("./SRT-MS-050__1_.txt")                 # Spectralon certificate
OUT       = Path("./reflectance_out")

DAY1_PLATES = ["AB100", "CH100", "RB100", "A", "B", "C"]
DAY2_PLATES = ["RB[50:50]", "RB[60:40]", "RB[70:30]", "RB[80:20]", "RB[90:10]"]

# how a filename maps to a day. Day 1 = 2026-09-01, day 2 = 2026-09-04 in your filenames.
DAY_RULES = [("2026-09-01", 1, DAY1_PLATES), ("2026-09-04", 2, DAY2_PLATES)]

ROUND_WL  = 0        # decimals for the wavelength labels: 0 -> 952, 958, 963 ...  1 -> 952.2
ERODE_PX  = 8        # pixels trimmed inward from each plate edge (rim and shadow)
SAVE_CUBES = False   # also write full reflectance ENVI cubes (~880 MB each -- off by default)
# -------------------------------------------------------------------------------

OUT.mkdir(exist_ok=True, parents=True)
print("data dir exists:", DATA_DIR.exists(), "| panel file exists:", PANEL_TXT.exists())

## 2. Reading the cubes

A plain `numpy.memmap` reader, so there is no extra dependency and the header handling is visible.

Your header declares `header offset = 2666496`, which is not a whole number of BIL lines. That is unusual
enough that the reader checks it against the real file size rather than trusting it, and tells you which
interpretation it used. A wrong offset produces a cube that looks like noise, so it is worth catching here.

Nothing is loaded into RAM until a region is sliced.

In [ ]:
ENVI_DTYPES = {1:"u1", 2:"i2", 3:"i4", 4:"f4", 5:"f8", 12:"u2", 13:"u4"}

def parse_envi_header(p):
    txt = Path(p).read_text(errors="ignore").replace("\r", "")
    body = txt.split("\n", 1)[1] if "\n" in txt else ""
    out, pos = {}, 0
    pat = re.compile(r"^\s*([\w\s]+?)\s*=\s*(\{.*?\}|[^\n]*)", re.S | re.M)
    while pos < len(body):
        m = pat.search(body, pos)
        if not m: break
        k, v = m.group(1).strip().lower(), m.group(2).strip()
        if v.startswith("{") and not v.endswith("}"):
            e = body.find("}", m.end(2)); v = body[m.start(2):e+1]; pos = e + 1
        else:
            pos = m.end(2)
        out[k] = v
    for k in ["samples", "lines", "bands", "header offset", "data type", "byte order"]:
        if k in out: out[k] = int(float(out[k]))
    if "data ignore value" in out: out["data ignore value"] = float(out["data ignore value"])
    if "wavelength" in out:
        out["wavelength"] = np.array([float(x) for x in
            out["wavelength"].strip("{}").replace("\n", " ").split(",") if x.strip()])
    out["interleave"] = out.get("interleave", "bsq").strip().lower()
    return out


class Cube:
    def __init__(self, hdr):
        self.hdr_path = Path(hdr); self.h = parse_envi_header(hdr)
        stem = self.hdr_path.with_suffix("")
        self.path = next((c for e in (".img", ".raw", ".dat", ".bin", "")
                          if (c := stem.with_suffix(e)).exists() and c != self.hdr_path), None)
        if self.path is None:
            raise FileNotFoundError(f"no data file beside {self.hdr_path.name}")
        L, S, B = self.h["lines"], self.h["samples"], self.h["bands"]
        dtc = ENVI_DTYPES[self.h["data type"]]
        nb = L * S * B * np.dtype(dtc).itemsize
        fs = self.path.stat().st_size
        dec = self.h.get("header offset", 0)
        if   fs == dec + nb: off, why = dec, "declared offset matches file size"
        elif fs == nb:       off, why = 0, "no offset (file size == data size)"
        elif fs > nb and fs - nb < 10_000_000:
            off, why = fs - nb, f"offset inferred as {fs - nb}"
        else:
            raise ValueError(f"{self.path.name}: size {fs}, expected {nb} data bytes "
                             f"(+{dec} offset). File may be truncated.")
        self.offset, self.offset_reason = off, why
        dt = np.dtype(dtc).newbyteorder("<" if self.h.get("byte order", 0) == 0 else ">")
        shp = {"bil": (L, B, S), "bsq": (B, L, S), "bip": (L, S, B)}[self.h["interleave"]]
        mm = np.memmap(self.path, dtype=dt, mode="r", offset=off, shape=shp)
        self.arr = {"bil": mm.transpose(0, 2, 1), "bsq": mm.transpose(1, 2, 0),
                    "bip": mm}[self.h["interleave"]]
        self.wl = self.h["wavelength"]
        self.ignore = self.h.get("data ignore value", None)

    @property
    def shape(self): return self.arr.shape

    def region(self, r0, r1, c0=0, c1=None):
        c1 = self.shape[1] if c1 is None else c1
        # np.array (not asarray) forces a writable copy -- a memmap slice is read-only
        b = np.array(self.arr[r0:r1, c0:c1, :], dtype=np.float32)
        if self.ignore is not None: b[b == self.ignore] = np.nan
        return b

    def brightness(self, step=2, lo=1500, hi=1700):
        idx = [i for i, w in enumerate(self.wl) if lo <= w <= hi][:8]
        s = np.array(self.arr[::step, ::step, :][:, :, idx], dtype=np.float32)
        if self.ignore is not None: s[s == self.ignore] = np.nan
        return np.nanmean(s, axis=2), step


CUBES, INFO = {}, {}
for h in sorted(DATA_DIR.rglob("*.hdr")):
    try:
        c = Cube(h); CUBES[h.stem] = c
        day, plates = next(((d, pl) for s, d, pl in DAY_RULES if s in h.stem), (None, []))
        INFO[h.stem] = dict(day=day, plates=plates)
        print(f"  {h.stem}")
        print(f"     {c.shape[0]} x {c.shape[1]} x {c.shape[2]} | {c.offset_reason} | "
              f"day {day}, {len(plates)} plates")
    except Exception as e:
        print(f"  !! {h.name}: {e}")

if not CUBES:
    raise SystemExit("no cubes opened -- check DATA_DIR")
no_day = [k for k, v in INFO.items() if v["day"] is None]
if no_day:
    print("\nNo day rule matched:", no_day, "-> add a filename substring to DAY_RULES")

## 3. Wavelengths and the certificate

The certificate is sampled every 1 nm from 250 to 2500 nm and gets interpolated onto the camera's 288 band
centres. Only the white column is used, as you asked.

Rounding to whole nanometres is safe here: band spacing is ~5.45 nm, so every rounded value stays unique
and no two bands collapse onto the same label.

One caveat worth recording: **four bands sit above 2500 nm**, past the end of the certificate
(2500.3, 2505.7, 2511.2, 2516.6 nm). Interpolation holds the last certified value flat across them, so
their reflectance is mildly extrapolated. They are flagged in the output rather than silently dropped.

In [ ]:
def load_white(txt_path, wl):
    rows = []
    for line in Path(txt_path).read_text(errors="ignore").replace("\r", "").split("\n"):
        p = line.strip().split("\t")
        if len(p) >= 2:
            try: rows.append([float(x) for x in p[:5]])
            except ValueError: continue
    a = np.array(rows)
    if a.size == 0: raise ValueError("could not parse the certificate")
    white = np.interp(wl, a[:, 0], a[:, 1])
    outside = (wl < a[0, 0]) | (wl > a[-1, 0])
    return white, outside, a

k0 = list(CUBES)[0]
WL = CUBES[k0].wl.copy()
for k, c in CUBES.items():
    assert np.allclose(c.wl, WL), f"{k} has a different band list -- handle it separately"

R_WHITE, OUTSIDE, PANEL_RAW = load_white(PANEL_TXT, WL)
WL_R = np.round(WL, ROUND_WL)
if ROUND_WL == 0: WL_R = WL_R.astype(int)
assert len(set(WL_R.tolist())) == WL.size, "rounding collapsed two bands -- raise ROUND_WL"

print(f"{WL.size} bands, {WL[0]:.2f}-{WL[-1]:.2f} nm")
print("rounded labels:", WL_R[:5], "...", WL_R[-5:])
print(f"white reflectance: {R_WHITE[0]:.4f} at {WL_R[0]} nm -> {R_WHITE[-1]:.4f} at {WL_R[-1]} nm")
print(f"bands beyond the certificate: {OUTSIDE.sum()} -> {WL_R[OUTSIDE]}")

plt.figure(figsize=(7, 2.6))
plt.plot(WL, R_WHITE, lw=1.4)
plt.scatter(WL[OUTSIDE], R_WHITE[OUTSIDE], s=14, color="crimson", zorder=3, label="extrapolated")
plt.xlabel("wavelength (nm)"); plt.ylabel("certified reflectance")
plt.title("white Spectralon, interpolated to the camera bands")
plt.legend(fontsize=7); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. Finding the white strip

There are two Spectralon strips at the top of each scan. Spectralon at 99 % is by far the brightest thing
in the frame — the powder sits around 40 % — so the white one is found as the brightest flat run in a
median profile down the scan. The median over the central columns is used rather than the mean, so a
specular speck cannot pull the answer.

The detected band is then trimmed at both ends so no row straddles the strip boundary or the darker
second strip.

**Check the plot before continuing.** If it has locked onto the wrong strip, put the row bounds into
`WHITE_MANUAL` and rerun — that overrides the detection completely.

In [ ]:
WHITE_MANUAL = {}          # e.g. {"stir1_...": (110, 195)}   rows in original pixel coordinates
WHITE_FRAC   = 0.88        # a row counts as "white strip" above this fraction of the profile peak
WHITE_MARGIN = 0.18        # fraction trimmed off each end of the detected run

def find_white_strip(cube, key):
    if key in WHITE_MANUAL:
        return tuple(WHITE_MANUAL[key]), None, None
    br, st = cube.brightness(step=2)
    S = br.shape[1]
    prof = np.nanmedian(br[:, int(S*0.2):int(S*0.8)], axis=1)
    mx = np.nanmax(prof)
    on = prof > WHITE_FRAC * mx
    runs, s = [], None
    for i, v in enumerate(np.append(on, False)):
        if v and s is None: s = i
        elif not v and s is not None:
            runs.append((s, i)); s = None
    if not runs: raise RuntimeError(f"{key}: no bright strip found")
    a, b = max(runs, key=lambda q: q[1] - q[0])
    pad = max(1, int((b - a) * WHITE_MARGIN))
    return ((a + pad) * st, (b - pad) * st), prof, st


WHITE_ROWS = {}
for key, cube in CUBES.items():
    rows, prof, st = find_white_strip(cube, key)
    WHITE_ROWS[key] = rows
    src = "manual" if key in WHITE_MANUAL else "auto"
    print(f"{key}: white strip rows {rows[0]}-{rows[1]}  ({rows[1]-rows[0]} lines, {src})")
    if prof is None: continue
    fig, ax = plt.subplots(figsize=(9, 2.4))
    ax.plot(np.arange(prof.size) * st, prof, lw=1.1)
    ax.axvspan(rows[0], rows[1], color="tab:orange", alpha=.35, label="white strip used")
    ax.set_xlabel("line (along-track)"); ax.set_ylabel("median radiance")
    ax.set_title(f"{key}: brightness down the scan"); ax.legend(fontsize=7); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(OUT / f"{key}_white_strip.png", dpi=120, bbox_inches="tight")
    plt.show()

In [ ]:
# Per-column white reference. Columns where the strip is missing or abnormally dark are
# repaired from their neighbours rather than left to produce wild reflectance values.
def build_white(cube, rows, bad_frac=0.5):
    r0, r1 = rows
    W = np.nanmedian(cube.region(r0, r1), axis=0)          # (samples, bands)
    ref = np.nanmedian(W, axis=0)                           # typical white per band
    lvl = np.nanmedian(W / np.maximum(ref, 1e-9), axis=1)   # per-column relative level
    bad = ~np.isfinite(lvl) | (lvl < bad_frac)
    if bad.any():
        good = np.where(~bad)[0]
        for j in range(W.shape[1]):
            W[bad, j] = np.interp(np.where(bad)[0], good, W[good, j])
    return W, int(bad.sum())


WHITE = {}
for key, cube in CUBES.items():
    W, nbad = build_white(cube, WHITE_ROWS[key])
    WHITE[key] = W
    print(f"{key}: white reference built, {nbad} of {W.shape[0]} columns repaired")

j = int(np.argmin(np.abs(WL - 1650)))
plt.figure(figsize=(7, 2.6))
for key, W in WHITE.items():
    plt.plot(W[:, j] / np.nanmax(W[:, j]), lw=1.1, label=key[:26])
plt.xlabel("across-track column"); plt.ylabel("relative white response")
plt.title("illumination profile across the swath at 1650 nm")
plt.legend(fontsize=6); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(OUT / "illumination_profile.png", dpi=120, bbox_inches="tight"); plt.show()
print("\nThe droop at the edges is lens vignetting. Dividing per column is what removes it;")
print("a single averaged white spectrum would leave it in the plate spectra.")

## 5. Finding the plates

Everything below the reference strip is either plate or table. Otsu's threshold separates the bright
powder from the dark background, and the plates sit in a single along-track column with clear gaps, so
runs of occupied rows give one box per plate.

They are then numbered in scan order and matched to your labels, starting from the end nearest the
reference strip.

**If the count comes out wrong**, put the row cut-points into `MANUAL_CUTS` for that scan.

In [ ]:
MANUAL_CUTS = {}        # e.g. {"stir1_...": [560, 830, 1100, 1370, 1650]}

def otsu(v):
    v = v[np.isfinite(v)]
    hist, edges = np.histogram(v, bins=256); p = hist / hist.sum(); c = np.cumsum(p)
    mids = (edges[:-1] + edges[1:]) / 2
    mu = np.cumsum(p * mids); mut = mu[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        sb = (mut * c - mu) ** 2 / (c * (1 - c))
    return mids[int(np.nanargmax(sb))]


def find_plates(cube, key, min_gap=5):
    br, st = cube.brightness(step=2)
    b = np.nan_to_num(br, nan=0.0).copy()
    ex = int((WHITE_ROWS[key][1] + 30) / st)
    b[:ex] = 0
    t = otsu(b[ex:])
    m = ndimage.binary_closing(ndimage.binary_opening(b > t, np.ones((3, 3))), np.ones((3, 3)))
    m = ndimage.binary_fill_holes(m)

    if key in MANUAL_CUTS:
        edges = [ex] + [int(c / st) for c in MANUAL_CUTS[key]] + [m.shape[0]]
        spans = list(zip(edges[:-1], edges[1:]))
    else:
        occ = m.sum(axis=1); thr = max(3, 0.08 * occ.max())
        on = occ > thr
        spans, s = [], None
        for i, v in enumerate(np.append(on, False)):
            if v and s is None: s = i
            elif not v and s is not None:
                if i - s >= min_gap: spans.append((s, i))
                s = None
    boxes = []
    for a, bb in spans:
        sub = m[a:bb]
        if sub.sum() < 50: continue
        cols = np.where(sub.any(axis=0))[0]
        boxes.append((a * st, bb * st, cols[0] * st, (cols[-1] + 1) * st))
    return boxes, m, st


PLATES = {}
for key, cube in CUBES.items():
    boxes, m, st = find_plates(cube, key)
    n = len(INFO[key]["plates"])
    PLATES[key] = boxes
    flag = "OK" if len(boxes) == n else f"!! expected {n}"
    print(f"{key}: {len(boxes)} plates found  {flag}")
    for i, (r0, r1, c0, c1) in enumerate(boxes, 1):
        lab = INFO[key]["plates"][i-1] if i <= n else "?"
        print(f"     {i}. {lab:10s} rows {r0:5d}-{r1:5d}  cols {c0:4d}-{c1:4d}")

## 6. Plate masks

The box is not the mask. Inside each one sits the black rim of the dish, a thick shadow crescent on the
leading edge, some table in the corners, and the occasional specular speck. The shadow is the one that
does real damage: it is a multiplicative darkening, so leaving it in shifts a spectrum in the same
direction a composition change would.

Per plate: Otsu inside the box, keep the largest connected region, fill holes, erode by `ERODE_PX`, then
drop pixels above the 99.7th percentile as specular.

The retained fraction is reported. Expect roughly 50–70 % of the box — the box includes rim and shadow, so
keeping nearly all of it would be the warning sign.

In [ ]:
def plate_mask(br, st, box, erode=ERODE_PX, spec_q=99.7):
    r0, r1, c0, c1 = [int(v / st) for v in box]
    sub = np.nan_to_num(br[r0:r1, c0:c1], nan=0.0)
    if sub.size < 50: return None
    m = sub > otsu(sub)
    m = ndimage.binary_opening(m, np.ones((3, 3)))
    lab, n = ndimage.label(m)
    if n == 0: return None
    m = lab == (1 + int(np.argmax(ndimage.sum(m, lab, range(1, n + 1)))))
    m = ndimage.binary_fill_holes(m)
    e = max(1, int(round(erode / st)))
    m = ndimage.binary_erosion(m, np.ones((3, 3)), iterations=e)
    if m.sum() > 50:
        hi = np.percentile(sub[m], spec_q)
        m &= ~ndimage.binary_dilation(sub > hi, np.ones((3, 3)))
    if m.sum() < 30: return None
    return dict(mask=m, box_dec=(r0, r1, c0, c1), frac=float(m.mean()), n=int(m.sum()))


MASKS = {}
for key, cube in CUBES.items():
    br, st = cube.brightness(step=2)
    out = []
    for i, box in enumerate(PLATES[key], 1):
        d = plate_mask(br, st, box)
        out.append(d)
        if d is None: print(f"{key} plate {i}: mask failed")
    MASKS[key] = (out, st)
    fr = [d["frac"] for d in out if d]
    print(f"{key}: kept " + " ".join(f"{f*100:.0f}%" for f in fr))

In [ ]:
def review(key, cols=3):
    cube = CUBES[key]; br, st = cube.brightness(step=2)
    out, _ = MASKS[key]; labels = INFO[key]["plates"]
    items = [(labels[i] if i < len(labels) else f"#{i+1}", d)
             for i, d in enumerate(out) if d]
    rows = int(np.ceil(len(items) / cols))
    fig, axs = plt.subplots(rows, cols, figsize=(3.4*cols, 2.7*rows), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for j, (lab, d) in enumerate(items):
        r, c = divmod(j, cols); ax = axs[r][c]
        r0, r1, c0, c1 = d["box_dec"]
        sub = np.nan_to_num(br[r0:r1, c0:c1])
        lo, hi = np.nanpercentile(sub, [2, 98])
        g = np.clip((sub - lo) / max(hi - lo, 1e-9), 0, 1)
        rgb = np.dstack([g, g, g])
        mm = d["mask"]
        rgb[..., 0] = np.where(mm, np.clip(g*.5+.5, 0, 1), g*.8)
        rgb[..., 1] = np.where(mm, g*.42, g*.8)
        rgb[..., 2] = np.where(mm, g*.32, g*.8)
        ax.imshow(np.clip(rgb, 0, 1)); ax.axis("off")
        ax.set_title(f"{lab}  {d['n']} px ({d['frac']*100:.0f}%)", fontsize=8)
    fig.suptitle(f"{key}: mask review (red = kept)", fontsize=9)
    plt.tight_layout(); plt.savefig(OUT / f"{key}_masks.png", dpi=125, bbox_inches="tight"); plt.show()

for key in CUBES: review(key)

## 7. Conversion

The division is applied inside each plate mask only — converting the whole 880 MB cube when most of the
frame is table would just burn memory.

The mask is at half resolution (the brightness image is decimated by 2), so it is expanded back to full
resolution before use.

Reflectance outside −0.05 to 1.6 is set to NaN: that range only contains pixels that were saturated or
in deep shadow, and they would otherwise distort the means.

In [ ]:
def plate_reflectance(key, idx):
    cube = CUBES[key]; out, st = MASKS[key]
    d = out[idx]
    if d is None: return None
    r0, r1, c0, c1 = [v * st for v in d["box_dec"]]
    r1 = min(r1, cube.shape[0]); c1 = min(c1, cube.shape[1])
    full = np.repeat(np.repeat(d["mask"], st, axis=0), st, axis=1)[:r1-r0, :c1-c0]
    L = cube.region(r0, r1, c0, c1)
    W = WHITE[key][c0:c1, :][None, :, :]
    R = L / np.maximum(W, 1e-9) * R_WHITE[None, None, :]
    R[(R < -0.05) | (R > 1.6)] = np.nan
    return R, full


rows, spectra = [], []
for key, cube in CUBES.items():
    labels = INFO[key]["plates"]
    for i, lab in enumerate(labels):
        if i >= len(MASKS[key][0]): break
        res = plate_reflectance(key, i)
        if res is None: continue
        R, m = res
        flat = R[m]
        mu, sd = np.nanmean(flat, axis=0), np.nanstd(flat, axis=0)
        rows.append(dict(scan=key, day=INFO[key]["day"], plate_no=i+1, label=lab,
                         n_pixels=int(np.isfinite(flat[:, j]).sum())))
        spectra.append((mu, sd))
        print(f"{key[:30]:30s} {lab:10s} {int(m.sum()):7d} px  "
              f"R@1650={mu[j]:.4f}  R@2310={mu[int(np.argmin(np.abs(WL-2310)))]:.4f}")

META = pd.DataFrame(rows)
MU = np.array([s[0] for s in spectra]); SD = np.array([s[1] for s in spectra])
print(f"\n{len(META)} plates converted")

## 8. Output

Three files:

- **`reflectance_mean.csv`** — one row per plate, one column per band, headed by the rounded wavelength.
  This is the main table.
- **`reflectance_sd.csv`** — the within-plate standard deviation, same shape. Useful for judging how
  uniform each powder surface was.
- **`reflectance_long.csv`** — the same data in long form (scan, day, label, wavelength, reflectance),
  for plotting tools that prefer tidy data.

`wavelengths.csv` records the exact band centres against their rounded labels, so the rounding stays
traceable and the four extrapolated bands are marked.

In [ ]:
cols = [str(w) for w in WL_R]

wide = pd.concat([META.reset_index(drop=True),
                  pd.DataFrame(np.round(MU, 6), columns=cols)], axis=1)
wide.to_csv(OUT / "reflectance_mean.csv", index=False)

sd_wide = pd.concat([META.reset_index(drop=True),
                     pd.DataFrame(np.round(SD, 6), columns=cols)], axis=1)
sd_wide.to_csv(OUT / "reflectance_sd.csv", index=False)

long = wide.melt(id_vars=list(META.columns), var_name="wavelength_nm", value_name="reflectance")
long["wavelength_nm"] = long["wavelength_nm"].astype(float)
long.sort_values(["scan", "plate_no", "wavelength_nm"]).to_csv(
    OUT / "reflectance_long.csv", index=False)

pd.DataFrame({"band": np.arange(1, WL.size+1), "wavelength_exact_nm": np.round(WL, 4),
              "wavelength_rounded_nm": WL_R, "white_certified": np.round(R_WHITE, 5),
              "beyond_certificate": OUTSIDE}).to_csv(OUT / "wavelengths.csv", index=False)

np.savez_compressed(OUT / "reflectance.npz", mean=MU, sd=SD, wl=WL, wl_rounded=WL_R,
                    labels=META["label"].values, scans=META["scan"].values,
                    days=META["day"].values)

print("written to", OUT.resolve())
for f in sorted(OUT.glob("*.csv")): print("  ", f.name)
print()
print(wide.iloc[:, :9].to_string(index=False))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for i, r in META.iterrows():
    ax = axs[0] if r["day"] == 1 else axs[1]
    ax.plot(WL, MU[i], lw=1.1, label=r["label"])
for ax, t in zip(axs, ["day 1", "day 2"]):
    for nm, txt in [(1450, "O-H"), (1730, "C-H lipid"), (1940, "O-H"),
                    (2100, "C-O carb."), (2310, "C-H comb.")]:
        ax.axvline(nm, color="0.75", lw=.8, ls=":")
        ax.text(nm, ax.get_ylim()[1], txt, fontsize=6, rotation=90, va="top")
    ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("reflectance"); ax.grid(alpha=.3)
    ax.set_title(f"{t}: mean plate reflectance")
    h, _ = ax.get_legend_handles_labels()
    if h: ax.legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.savefig(OUT / "reflectance_spectra.png", dpi=130, bbox_inches="tight"); plt.show()

print("Sanity check on the result:")
print("  Reflectance should sit roughly 0.2-0.6 across the range, with clear dips near 1450 and")
print("  1940 nm (water) and near 2300 nm (C-H).")
print("  Pure robusta and pure chicory should separate around 1730 nm -- coffee carries 10-15% oil,")
print("  chicory almost none. If RB100 and CH100 overlay almost exactly, suspect the conversion")
print("  rather than the coffee: usually the white strip rows are wrong.")

## 9. Optional — full reflectance cubes

Off by default. Each output is the same size as its input (~880 MB), and for most purposes the per-plate
table above is what you actually want. Turn `SAVE_CUBES` on in the config cell if you need the imagery,
for example to run spatial analysis or hand the cubes to other software.

They are written as ENVI BIL float32 with a header carrying the same band list, in chunks of lines so
memory stays flat regardless of scan length.

In [ ]:
def write_reflectance_cube(key, chunk=200):
    cube = CUBES[key]
    L, S, B = cube.shape
    out_img = OUT / f"{key}_reflectance.img"
    W = WHITE[key][None, :, :]
    with open(out_img, "wb") as f:
        for r0 in range(0, L, chunk):
            r1 = min(L, r0 + chunk)
            R = cube.region(r0, r1) / np.maximum(W, 1e-9) * R_WHITE[None, None, :]
            R[(R < -0.05) | (R > 1.6)] = np.nan
            np.ascontiguousarray(R.transpose(0, 2, 1).astype(np.float32)).tofile(f)
    hdr = ["ENVI", "description = { reflectance, white-Spectralon referenced, per-column }",
           f"samples = {S}", f"lines = {L}", f"bands = {B}", "header offset = 0",
           "data type = 4", "interleave = bil", "byte order = 0", "wavelength units = nm",
           "wavelength = {" + ", ".join(f"{w:.6f}" for w in WL) + "}"]
    (OUT / f"{key}_reflectance.hdr").write_text("\n".join(hdr))
    return out_img

if SAVE_CUBES:
    for key in CUBES:
        p = write_reflectance_cube(key)
        print(f"wrote {p.name}  ({p.stat().st_size/1e6:.0f} MB)")
else:
    print("SAVE_CUBES is False -- per-plate tables only. Set it True in the config cell if you")
    print("need full reflectance imagery (about 880 MB per scan).")